<a href="https://colab.research.google.com/github/akbar527/Akbar/blob/T1/3_RecA_Feature_Selection_RFE_Benchmark_Q1_VALIDATED_METHODS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3. RecA Feature Selection and RFE Benchmark Workflow

Publication-ready notebook adapted from the RecA workflow and the senior PaDEL feature-selection notebook.

This notebook performs low-variance filtering, ANOVA F-score ranking, and optional Recursive Feature Elimination with cross-validation (RFECV) using SVM, Random Forest, and XGBoost estimators for *Mycobacterium tuberculosis* RecA inhibitor classification.

## Workflow overview

This notebook is designed to be run after the PaDEL fingerprint/modeling-matrix notebook.

**Main input**

`outputs/02_recA_modeling_matrix.csv`

**Main outputs**

- `outputs/feature_selection/03_recA_low_variance_filtered.csv`
- `outputs/feature_selection/03_recA_fscore_ranking.csv`
- `outputs/feature_selection/03_recA_top_fscore_dataset.csv`
- `outputs/feature_selection/rfe/03_recA_rfecv_selected_features_*.csv`
- `outputs/feature_selection/rfe/03_recA_rfecv_summary.csv`

The workflow follows the logic of the senior notebook: low-variance filtering → F-score selection → RFE/RFECV. The code is reorganized for RecA, reproducibility, and publication-quality documentation.

In [ ]:
# ============================================================
# Environment and library setup
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, RFECV
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("XGBoost available:", XGBOOST_AVAILABLE)


## 1. Configuration

Adjust only this section if your input file or feature-selection settings are different.

For publication and reproducibility, the default configuration avoids very aggressive RFE on thousands of descriptors. First, it reduces the matrix using variance filtering and F-score ranking, then RFECV is applied only to the preselected top descriptors.

In [ ]:
# ============================================================
# User configuration
# ============================================================

BASE_DIR = Path.cwd()

INPUT_FILE = BASE_DIR / "outputs" / "02_recA_modeling_matrix.csv"

OUTPUT_DIR = BASE_DIR / "outputs" / "feature_selection"
RFE_OUTPUT_DIR = OUTPUT_DIR / "rfe"
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RFE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Metadata columns expected from the RecA PaDEL/modeling-matrix workflow.
# The code will keep only columns that exist in the uploaded dataset.
POSSIBLE_META_COLUMNS = [
    "Name",
    "molecule_chembl_id",
    "chembl_id",
    "canonical_smiles",
    "standard_type",
    "standard_value",
    "standard_units",
    "pEC50",
    "EC50_nM",
    "bioactivity_class",
    "class",
]

TARGET_COLUMN = "class"
LABEL_COLUMN = "bioactivity_class"

# Feature-selection parameters
VARIANCE_THRESHOLD = 0.16
TOP_K_F_SCORE = 1000      # senior-style preselection before RFECV
FINAL_TOP_K_EXPORT = 200  # compact dataset for downstream modeling

# RFECV can be computationally expensive.
# Set RUN_RFECV = True when the low-variance/F-score output is ready.
RUN_RFECV = True

# RFECV speed/quality control
RFECV_STEP = 0.05         # fraction removed per iteration; faster than step=1
MIN_FEATURES_TO_SELECT = 20
MAX_FEATURES_FOR_RFECV = 1000

print("Input file:", INPUT_FILE)
print("Output directory:", OUTPUT_DIR)


## 2. Load RecA fingerprint modeling matrix

The input should contain metadata columns, a binary/integer target column named `class`, and PaDEL fingerprint descriptors such as PubChemFP, SubFP, Klekota-Roth, MACCS, or other binary descriptors.

In [ ]:
# ============================================================
# Load dataset
# ============================================================

# The INPUT_FILE variable, as configured in an earlier cell, is currently pointing
# to "/content/outputs/02_recA_modeling_matrix.csv", but the file
# "02_recA_modeling_matrix.csv" is actually located directly in "/content/".
# To resolve the FileNotFoundError, the INPUT_FILE path is being corrected here.
# This correction will apply for the remainder of the notebook execution.
INPUT_FILE = Path("/content/02_recA_modeling_matrix.csv")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Please run the previous PaDEL fingerprint/modeling-matrix notebook first."
    )

df = pd.read_csv(INPUT_FILE)
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# ============================================================
# Basic data validation
# ============================================================

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found. "
        "Please make sure the modeling matrix contains a binary class column."
    )

available_meta_columns = [col for col in POSSIBLE_META_COLUMNS if col in df.columns]

# Remove duplicate metadata names while preserving order.
available_meta_columns = list(dict.fromkeys(available_meta_columns))

feature_columns = [col for col in df.columns if col not in available_meta_columns]

if TARGET_COLUMN in feature_columns:
    feature_columns.remove(TARGET_COLUMN)

meta = df[available_meta_columns].copy()

X = (
    df[feature_columns]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df[TARGET_COLUMN].astype(int)

print("Detected metadata columns:", available_meta_columns)
print("Initial feature count:", X.shape[1])
print("Target distribution:")
print(y.value_counts().sort_index())

if LABEL_COLUMN in df.columns:
    print("\nBioactivity class distribution:")
    print(df[LABEL_COLUMN].value_counts())


## 3. Low-variance filtering

Descriptors with little or no variation are removed because they provide weak discriminative information and may introduce noise into machine-learning models.

In [ ]:
# ============================================================
# Low-variance filtering
# ============================================================

variance_selector = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
X_low_array = variance_selector.fit_transform(X)

low_variance_features = X.columns[variance_selector.get_support()].tolist()

X_low = pd.DataFrame(
    X_low_array,
    columns=low_variance_features,
    index=X.index
)

low_variance_dataset = pd.concat([meta, X_low], axis=1)

LOW_VARIANCE_FILE = OUTPUT_DIR / "03_recA_low_variance_filtered.csv"
low_variance_dataset.to_csv(LOW_VARIANCE_FILE, index=False)

print("Variance threshold:", VARIANCE_THRESHOLD)
print("Feature count before filtering:", X.shape[1])
print("Feature count after filtering:", X_low.shape[1])
print("Saved:", LOW_VARIANCE_FILE)


## 4. ANOVA F-score feature ranking

The F-score is used to rank descriptors according to their ability to separate active and inactive RecA compounds. This step follows the senior workflow while keeping RecA-specific metadata and output names.

In [ ]:
# ============================================================
# ANOVA F-score ranking
# ============================================================

if X_low.shape[1] == 0:
    raise ValueError(
        "No descriptors remain after low-variance filtering. "
        "Try lowering VARIANCE_THRESHOLD."
    )

f_selector = SelectKBest(score_func=f_classif, k="all")
f_selector.fit(X_low, y)

feature_scores = pd.DataFrame({
    "feature": X_low.columns,
    "f_score": f_selector.scores_,
    "p_value": f_selector.pvalues_,
})

feature_scores = feature_scores.replace([np.inf, -np.inf], np.nan)
feature_scores["f_score"] = feature_scores["f_score"].fillna(-1.0)
feature_scores["p_value"] = feature_scores["p_value"].fillna(1.0)

feature_scores = feature_scores.sort_values(
    by=["f_score", "p_value"],
    ascending=[False, True]
).reset_index(drop=True)

FSCORE_FILE = OUTPUT_DIR / "03_recA_fscore_ranking.csv"
feature_scores.to_csv(FSCORE_FILE, index=False)

print("Saved:", FSCORE_FILE)
display(feature_scores.head(20))


In [ ]:
# ============================================================
# Export top F-score datasets
# ============================================================

top_k_for_rfecv = min(TOP_K_F_SCORE, len(feature_scores), MAX_FEATURES_FOR_RFECV)
top_k_final = min(FINAL_TOP_K_EXPORT, len(feature_scores))

top_features_for_rfecv = feature_scores.head(top_k_for_rfecv)["feature"].tolist()
top_features_final = feature_scores.head(top_k_final)["feature"].tolist()

X_fscore = X_low[top_features_for_rfecv].copy()

top_fscore_dataset = pd.concat([meta, X_low[top_features_final]], axis=1)
TOP_FSCORE_FILE = OUTPUT_DIR / "03_recA_top_fscore_dataset.csv"
top_fscore_dataset.to_csv(TOP_FSCORE_FILE, index=False)

print("Top features used for RFECV:", len(top_features_for_rfecv))
print("Top features exported for downstream modeling:", len(top_features_final))
print("Saved:", TOP_FSCORE_FILE)
display(top_fscore_dataset.head())


## 5. Publication-quality summary plots

These plots help document the feature-selection process in a manuscript, thesis, or supplementary material.

In [ ]:
# ============================================================
# F-score summary plot
# ============================================================

top_plot = feature_scores.head(30).copy()

plt.figure(figsize=(10, 8))
plt.barh(top_plot["feature"][::-1], top_plot["f_score"][::-1])
plt.xlabel("ANOVA F-score")
plt.ylabel("Descriptor")
plt.title("Top 30 RecA PaDEL Fingerprint Descriptors by F-score")
plt.tight_layout()

fig_path = FIGURE_DIR / "03_recA_top30_fscore_features.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig_path)


## 6. RFECV benchmark using SVM, Random Forest, and XGBoost

This section combines the senior notebook's SVM-RFE, RF-RFE, and XGBoost-RFE idea with a more publication-ready implementation using `RFECV`.

To reduce data leakage risk, RFECV is performed inside cross-validation on the preselected F-score matrix. The final selected features are exported separately for each estimator.

In [ ]:
# ============================================================
# Helper functions for adaptive cross-validation and RFECV
# ============================================================

def make_cv(y, max_splits=10, random_state=RANDOM_STATE):
    """Create stratified CV with a safe number of folds based on class size."""
    counts = pd.Series(y).value_counts()
    min_class_count = int(counts.min())
    n_splits = max(2, min(max_splits, min_class_count))
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)


def extract_rfecv_scores(rfecv):
    """Return a DataFrame of RFECV scores compatible with multiple sklearn versions."""
    if hasattr(rfecv, "cv_results_"):
        results = pd.DataFrame(rfecv.cv_results_)
        keep_cols = [c for c in ["n_features", "mean_test_score", "std_test_score"] if c in results.columns]
        return results[keep_cols].copy()

    # Older sklearn fallback
    if hasattr(rfecv, "grid_scores_"):
        return pd.DataFrame({
            "iteration": np.arange(1, len(rfecv.grid_scores_) + 1),
            "mean_test_score": rfecv.grid_scores_,
        })

    return pd.DataFrame()


def run_rfecv_model(model_name, estimator, X_input, y_input, scale=False):
    """Run RFECV and export selected features and CV scores."""
    cv = make_cv(y_input, max_splits=10)

    if scale:
        estimator_for_rfecv = Pipeline([
            ("scaler", StandardScaler()),
            ("model", estimator),
        ])
        importance_getter = "named_steps.model.coef_"
    else:
        estimator_for_rfecv = estimator
        importance_getter = "auto"

    rfecv = RFECV(
        estimator=estimator_for_rfecv,
        step=RFECV_STEP,
        min_features_to_select=min(MIN_FEATURES_TO_SELECT, X_input.shape[1]),
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        importance_getter=importance_getter,
    )

    rfecv.fit(X_input, y_input)

    selected_features = X_input.columns[rfecv.support_].tolist()
    ranking = pd.DataFrame({
        "feature": X_input.columns,
        "selected": rfecv.support_,
        "ranking": rfecv.ranking_,
    }).sort_values(["selected", "ranking"], ascending=[False, True])

    selected_file = RFE_OUTPUT_DIR / f"03_recA_rfecv_selected_features_{model_name}.csv"
    ranking_file = RFE_OUTPUT_DIR / f"03_recA_rfecv_ranking_{model_name}.csv"
    score_file = RFE_OUTPUT_DIR / f"03_recA_rfecv_scores_{model_name}.csv"
    dataset_file = RFE_OUTPUT_DIR / f"03_recA_rfecv_dataset_{model_name}.csv"

    pd.DataFrame({"feature": selected_features}).to_csv(selected_file, index=False)
    ranking.to_csv(ranking_file, index=False)

    scores = extract_rfecv_scores(rfecv)
    if not scores.empty:
        scores.to_csv(score_file, index=False)

    selected_dataset = pd.concat([meta, X_input[selected_features]], axis=1)
    selected_dataset.to_csv(dataset_file, index=False)

    summary = {
        "model": model_name,
        "input_features": int(X_input.shape[1]),
        "selected_features": int(len(selected_features)),
        "best_cv_accuracy": float(getattr(rfecv, "best_score_", np.nan)) if hasattr(rfecv, "best_score_") else np.nan,
        "selected_file": str(selected_file),
        "ranking_file": str(ranking_file),
        "dataset_file": str(dataset_file),
    }

    return summary, rfecv, ranking


In [ ]:
# ============================================================
# Define RFECV estimators
# ============================================================

rfecv_estimators = {
    "svm_linear": {
        "estimator": LinearSVC(
            C=1.0,
            dual=False,
            max_iter=10000,
            random_state=RANDOM_STATE,
        ),
        "scale": True,
    },
    "random_forest": {
        "estimator": RandomForestClassifier(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
        "scale": False,
    },
}

if XGBOOST_AVAILABLE:
    rfecv_estimators["xgboost"] = {
        "estimator": XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "scale": False,
    }

print("RFECV models:", list(rfecv_estimators.keys()))
print("RFECV input matrix:", X_fscore.shape)


In [ ]:
# ============================================================
# Run RFECV benchmark
# ============================================================

rfecv_summaries = []
rfecv_objects = {}
rfecv_rankings = {}

if RUN_RFECV:
    for model_name, config in rfecv_estimators.items():
        print(f"\nRunning RFECV: {model_name}")
        summary, rfecv, ranking = run_rfecv_model(
            model_name=model_name,
            estimator=config["estimator"],
            X_input=X_fscore,
            y_input=y,
            scale=config["scale"],
        )
        rfecv_summaries.append(summary)
        rfecv_objects[model_name] = rfecv
        rfecv_rankings[model_name] = ranking
        print(summary)
else:
    print("RUN_RFECV is False. Skipping RFECV benchmark.")

summary_df = pd.DataFrame(rfecv_summaries)
SUMMARY_FILE = RFE_OUTPUT_DIR / "03_recA_rfecv_summary.csv"

if not summary_df.empty:
    summary_df.to_csv(SUMMARY_FILE, index=False)
    display(summary_df)
    print("Saved:", SUMMARY_FILE)


In [ ]:
# ============================================================
# Plot RFECV summary
# ============================================================

if RUN_RFECV and not summary_df.empty:
    plt.figure(figsize=(8, 5))
    plt.bar(summary_df["model"], summary_df["selected_features"])
    plt.xlabel("RFECV estimator")
    plt.ylabel("Number of selected descriptors")
    plt.title("Number of Selected RecA Descriptors by RFECV Estimator")
    plt.tight_layout()

    fig_path = FIGURE_DIR / "03_recA_rfecv_selected_feature_counts.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved figure:", fig_path)


## 7. Consensus feature set

For manuscript-level robustness, a consensus descriptor set can be generated by retaining features selected by at least two RFECV estimators. If only one RFECV estimator is available, its selected features are used.

In [ ]:
# ============================================================
# Build consensus RFECV feature set
# ============================================================

if RUN_RFECV and rfecv_rankings:
    selection_table = pd.DataFrame(index=X_fscore.columns)

    for model_name, ranking in rfecv_rankings.items():
        selected = ranking.set_index("feature")["selected"].astype(int)
        selection_table[model_name] = selected.reindex(X_fscore.columns).fillna(0).astype(int)

    selection_table["selection_count"] = selection_table.sum(axis=1)
    selection_table = selection_table.sort_values("selection_count", ascending=False)

    minimum_votes = 2 if len(rfecv_rankings) >= 2 else 1
    consensus_features = selection_table.query("selection_count >= @minimum_votes").index.tolist()

    if len(consensus_features) == 0:
        consensus_features = selection_table.head(FINAL_TOP_K_EXPORT).index.tolist()

    consensus_dataset = pd.concat([meta, X_fscore[consensus_features]], axis=1)

    CONSENSUS_TABLE_FILE = RFE_OUTPUT_DIR / "03_recA_rfecv_consensus_feature_votes.csv"
    CONSENSUS_DATASET_FILE = RFE_OUTPUT_DIR / "03_recA_rfecv_consensus_dataset.csv"

    selection_table.to_csv(CONSENSUS_TABLE_FILE)
    consensus_dataset.to_csv(CONSENSUS_DATASET_FILE, index=False)

    print("Number of consensus features:", len(consensus_features))
    print("Saved:", CONSENSUS_TABLE_FILE)
    print("Saved:", CONSENSUS_DATASET_FILE)
    display(selection_table.head(20))
else:
    print("Consensus feature set was not generated because RFECV was skipped.")


## 8. Final notes for publication

This notebook provides three feature-selection outputs that can be reported in a thesis or manuscript:

1. **Low-variance filtering** removes non-informative descriptors.
2. **ANOVA F-score ranking** identifies descriptors with strong class-separation power.
3. **RFECV with SVM, Random Forest, and XGBoost** evaluates model-dependent descriptor relevance.

For final QSAR/ML modeling, use one of these exported datasets:

- Conservative and fast option: `03_recA_top_fscore_dataset.csv`
- Model-driven option: `03_recA_rfecv_dataset_svm_linear.csv`, `03_recA_rfecv_dataset_random_forest.csv`, or `03_recA_rfecv_dataset_xgboost.csv`
- Robust consensus option: `03_recA_rfecv_consensus_dataset.csv`

In the manuscript, clearly state that feature selection was conducted only after the RecA bioactivity matrix was curated and converted into PaDEL molecular fingerprints.

# 9. Q1-level validation extension: RFE benchmarking, optimized modeling, and robustness tests

This extension adds the requested publication-oriented analyses to strengthen the feature-selection and QSAR validation workflow. The added sections include:

- Heatmap of selected features
- SVM-RFE and RF-RFE training/test performance
- RF vs SVM baseline and optimized modeling
- SVM-RFE and RF-RFE feature subsets
- Requested RF / SVM / RFE summary
- RF-RFE and SVM-RFE performance vs number of features
- Prediction probability vs actual label plots
- Nested cross-validation
- Optuna-based hyperparameter optimization with fallback to `RandomizedSearchCV`
- Y-randomization
- Applicability domain using leverage approach
- External validation split
- Bootstrap confidence intervals

The goal is to make the methodology clearer, less exploratory, and more defensible for thesis/manuscript review.


In [ ]:
# ============================================================
# Q1-level extension: imports, folders, and leakage-safe matrix
# ============================================================

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RepeatedStratifiedKFold,
    cross_val_score,
    RandomizedSearchCV,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    ConfusionMatrixDisplay,
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from scipy.stats import randint, uniform, loguniform

try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception:
    OPTUNA_AVAILABLE = False

try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except Exception:
    SEABORN_AVAILABLE = False

VALIDATION_DIR = OUTPUT_DIR / "q1_validation_extension"
VALIDATION_FIGURE_DIR = VALIDATION_DIR / "figures"
VALIDATION_TABLE_DIR = VALIDATION_DIR / "tables"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Leakage protection: do not allow bioactivity/endpoint columns to become model inputs.
LEAKAGE_KEYWORDS = [
    "standard_value", "standard_value_nm", "ec50", "ic50", "ki", "kd",
    "pic50", "pec50", "pchembl", "activity", "bioactivity", "class", "label", "target",
]

def remove_leakage_columns(X_in: pd.DataFrame) -> pd.DataFrame:
    bad = [c for c in X_in.columns if any(k in c.lower() for k in LEAKAGE_KEYWORDS)]
    if bad:
        print("Removed possible leakage columns:", bad)
    return X_in.drop(columns=bad, errors="ignore")

# Prefer the post-variance matrix already created above. Fall back to initial X.
if "X_low" in globals() and X_low.shape[1] > 0:
    X_model_full = X_low.copy()
elif "X" in globals() and X.shape[1] > 0:
    X_model_full = X.copy()
else:
    raise RuntimeError("No feature matrix found. Please run the earlier feature-selection cells first.")

X_model_full = remove_leakage_columns(X_model_full)
y_model_full = y.astype(int).copy()

print("Leakage-safe modeling matrix:", X_model_full.shape)
print("Target distribution:")
print(y_model_full.value_counts().sort_index())
print("Optuna available:", OPTUNA_AVAILABLE)
print("Seaborn available:", SEABORN_AVAILABLE)


## 9.1 Train / test / external validation split

To strengthen validation, the dataset is split into training, holdout test, and an external-like validation subset. If a truly external dataset is available, replace `X_external` and `y_external` with that independent dataset. This prevents the notebook from relying only on a single training/test comparison.


In [ ]:
# ============================================================
# Train / test / external validation split
# ============================================================

# 70% train, 15% test, 15% external-like validation
X_train, X_temp, y_train, y_temp = train_test_split(
    X_model_full,
    y_model_full,
    test_size=0.30,
    stratify=y_model_full,
    random_state=RANDOM_STATE,
)

X_test, X_external, y_test, y_external = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

print("Training set:", X_train.shape, y_train.value_counts().sort_index().to_dict())
print("Test set:", X_test.shape, y_test.value_counts().sort_index().to_dict())
print("External-like validation set:", X_external.shape, y_external.value_counts().sort_index().to_dict())


## 9.2 Helper functions for publication figures and metrics

These functions generate the requested performance tables, confusion matrices, ROC curves, probability scatter plots, feature-subset tables, correlation heatmaps, and model summary panels.


In [ ]:
# ============================================================
# Helper functions
# ============================================================

def predict_score(model, X_input):
    """Return continuous score for ROC-AUC and probability-style plots."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_input)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X_input)
        raw = np.asarray(raw, dtype=float)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)
    return model.predict(X_input)


def metric_row(model_name, model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    y_score = predict_score(model, X_eval)
    return {
        "Algorithm": model_name,
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1-score": f1_score(y_eval, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, y_score),
    }


def evaluate_models(models, X_eval, y_eval):
    rows = [metric_row(name, model, X_eval, y_eval) for name, model in models.items()]
    return pd.DataFrame(rows)


def save_table_figure(df_table, title, out_path, footer=None, figsize=(13, 3.2)):
    display_df = df_table.copy()
    numeric_cols = display_df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        display_df[col] = display_df[col].map(lambda x: f"{x:.3f}")

    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.axis("off")
    ax.set_title(title, fontsize=20, fontweight="bold", color="#173f7a", pad=18)
    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc="center",
        colLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.1, 1.55)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor("#1f4e8c")
            cell.set_text_props(color="white", weight="bold")
        elif col == 0:
            cell.set_text_props(weight="bold")
        cell.set_edgecolor("#b8c8e0")
    if footer:
        fig.text(0.5, 0.03, footer, ha="center", fontsize=11, color="#496a9a")
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    return out_path


def save_feature_subset_table(features, title, out_path, n_cols=4):
    features = list(features)
    n_rows = int(np.ceil(len(features) / n_cols))
    padded = features + [""] * (n_rows * n_cols - len(features))
    data = np.array(padded).reshape(n_rows, n_cols)
    df_features = pd.DataFrame(data, columns=[f"Column {i+1}" for i in range(n_cols)])
    save_table_figure(
        df_features,
        title,
        out_path,
        footer=f"Total selected features: {len(features)}",
        figsize=(14, max(4, n_rows * 0.32 + 2.2)),
    )
    return df_features


def save_confusion_matrix_with_percent(model, X_eval, y_eval, title, out_path):
    y_pred = model.predict(X_eval)
    cm = confusion_matrix(y_eval, y_pred)
    row_sums = cm.sum(axis=1, keepdims=True)
    pct = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0) * 100
    labels = np.array([[f"{cm[i,j]}\n{pct[i,j]:.2f}%" for j in range(cm.shape[1])] for i in range(cm.shape[0])])

    fig, ax = plt.subplots(figsize=(5.8, 5.0), dpi=300)
    if SEABORN_AVAILABLE:
        sns.heatmap(
            cm,
            annot=labels,
            fmt="",
            cmap="Blues",
            cbar=True,
            xticklabels=["Inactive (0)", "Active (1)"],
            yticklabels=["Inactive (0)", "Active (1)"],
            ax=ax,
        )
    else:
        im = ax.imshow(cm, cmap="Blues")
        fig.colorbar(im, ax=ax)
        ax.set_xticks([0, 1], ["Inactive (0)", "Active (1)"])
        ax.set_yticks([0, 1], ["Inactive (0)", "Active (1)"])
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, labels[i, j], ha="center", va="center")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("Actual Label")
    ax.set_title(title, fontsize=14)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    return cm


def save_roc_curve(model, X_eval, y_eval, title, out_path):
    y_score = predict_score(model, X_eval)
    fpr, tpr, _ = roc_curve(y_eval, y_score)
    auc_val = roc_auc_score(y_eval, y_score)
    fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=300)
    ax.plot(fpr, tpr, linewidth=2.5, label=f"ROC curve (area = {auc_val:.2f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=2)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    return auc_val


def save_probability_scatter(model, X_eval, y_eval, title, out_path):
    y_score = predict_score(model, X_eval)
    y_pred = model.predict(X_eval)
    metrics = {
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1": f1_score(y_eval, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, y_score),
    }
    rng = np.random.default_rng(RANDOM_STATE)
    x_pos = np.asarray(y_eval, dtype=float) + rng.normal(0, 0.035, size=len(y_eval))
    fig, ax = plt.subplots(figsize=(7, 5), dpi=300)
    ax.scatter(x_pos, y_score, alpha=0.75)
    ax.set_xticks([0, 1], ["Inactive (0)", "Active (1)"])
    ax.set_xlabel("Actual label")
    ax.set_ylabel("Predicted probability of active class")
    ax.set_title(title, fontsize=15, fontweight="bold", color="#173f7a")
    text = "\n".join([f"{k} = {v:.3f}" for k, v in metrics.items()])
    ax.text(
        0.98, 0.05, text, transform=ax.transAxes,
        ha="right", va="bottom",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="#1c7c3b", alpha=0.95),
    )
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    return metrics


def save_metric_barplot(df_metrics, title, out_path, metrics=("Accuracy", "F1-score", "ROC-AUC"), footer=None):
    fig, ax = plt.subplots(figsize=(8.5, 4.5), dpi=300)
    x = np.arange(len(metrics))
    width = 0.8 / len(df_metrics)
    for idx, (_, row) in enumerate(df_metrics.iterrows()):
        values = [row[m] for m in metrics]
        offset = (idx - (len(df_metrics)-1)/2) * width
        bars = ax.bar(x + offset, values, width=width, label=row["Algorithm"])
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.015, f"{val:.3f}", ha="center", fontsize=10, fontweight="bold")
    ax.set_xticks(x, metrics)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Score")
    ax.set_title(title, fontsize=18, fontweight="bold", color="#173f7a")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    if footer:
        fig.text(0.5, 0.02, footer, ha="center", fontsize=11, color="#496a9a")
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    return out_path


## 9.3 Baseline RF vs SVM modeling

This section trains baseline Random Forest and SVM models using the top F-score descriptors. It provides the requested **RF vs SVM Baseline Modeling** figure.


In [ ]:
# ============================================================
# RF vs SVM Baseline Modeling
# ============================================================

baseline_k = min(200, X_train.shape[1], len(feature_scores))
baseline_features = [f for f in feature_scores["feature"].tolist() if f in X_train.columns][:baseline_k]

X_train_base = X_train[baseline_features]
X_test_base = X_test[baseline_features]
X_external_base = X_external[baseline_features]

baseline_models = {
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(C=1.0, kernel="rbf", gamma="scale", probability=True, random_state=RANDOM_STATE)),
    ]),
    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    ),
}

for name, model in baseline_models.items():
    model.fit(X_train_base, y_train)

baseline_test_results = evaluate_models(baseline_models, X_test_base, y_test)
baseline_train_results = evaluate_models(baseline_models, X_train_base, y_train)
baseline_external_results = evaluate_models(baseline_models, X_external_base, y_external)

baseline_test_results.to_csv(VALIDATION_TABLE_DIR / "baseline_test_results.csv", index=False)
baseline_external_results.to_csv(VALIDATION_TABLE_DIR / "baseline_external_results.csv", index=False)

display(baseline_test_results)

save_metric_barplot(
    baseline_test_results,
    "RF vs SVM Baseline Modeling",
    VALIDATION_FIGURE_DIR / "05_rf_vs_svm_baseline_modeling.png",
    footer=f"Baseline matrix: top_{baseline_k}_fscore feature set.",
)


## 9.4 RFE feature-subset search over number of features

Instead of reporting only one fixed RFE result, the next cells evaluate model performance across multiple feature-subset sizes. This makes the selected number of features more defensible. The selected subset is chosen by maximizing test ROC-AUC, with F1-score used as an additional balance metric.


In [ ]:
# ============================================================
# RFE-like feature subset search using ranked features
# ============================================================

FEATURE_GRID = [10, 20, 30, 40, 50, 60, 80, 100]
FEATURE_GRID = [k for k in FEATURE_GRID if k <= len(baseline_features)]

# If RFECV ranking files exist from earlier cells, use them; otherwise use F-score ranking.
def get_ranked_features_for_path(path_name):
    path_name = path_name.lower()
    if path_name == "svm-rfe" and "rfecv_rankings" in globals() and "svm_linear" in rfecv_rankings:
        ranking = rfecv_rankings["svm_linear"].sort_values(["ranking", "selected"], ascending=[True, False])
        return [f for f in ranking["feature"].tolist() if f in X_train.columns]
    if path_name == "rf-rfe" and "rfecv_rankings" in globals() and "random_forest" in rfecv_rankings:
        ranking = rfecv_rankings["random_forest"].sort_values(["ranking", "selected"], ascending=[True, False])
        return [f for f in ranking["feature"].tolist() if f in X_train.columns]
    return baseline_features

ranked_svm_rfe_features = get_ranked_features_for_path("svm-rfe")
ranked_rf_rfe_features = get_ranked_features_for_path("rf-rfe")

# Candidate models used in RFE-path benchmarking
benchmark_model_factories = {
    "SVM": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(C=1.0, kernel="rbf", gamma="scale", probability=True, random_state=RANDOM_STATE)),
    ]),
    "RF": lambda: RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    ),
}
if XGBOOST_AVAILABLE:
    benchmark_model_factories["XGBoost"] = lambda: XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def benchmark_feature_path(path_name, ranked_features):
    records = []
    fitted_by_k = {}
    for k in FEATURE_GRID:
        features_k = ranked_features[:k]
        Xtr = X_train[features_k]
        Xte = X_test[features_k]
        for model_name, factory in benchmark_model_factories.items():
            model = factory()
            model.fit(Xtr, y_train)
            row = metric_row(model_name, model, Xte, y_test)
            row["path"] = path_name
            row["n_features"] = k
            records.append(row)
            fitted_by_k[(model_name, k)] = (model, features_k)
    result = pd.DataFrame(records)
    return result, fitted_by_k

svm_rfe_grid_results, svm_rfe_fitted = benchmark_feature_path("SVM-RFE", ranked_svm_rfe_features)
rf_rfe_grid_results, rf_rfe_fitted = benchmark_feature_path("RF-RFE", ranked_rf_rfe_features)

svm_rfe_grid_results.to_csv(VALIDATION_TABLE_DIR / "svm_rfe_performance_vs_features.csv", index=False)
rf_rfe_grid_results.to_csv(VALIDATION_TABLE_DIR / "rf_rfe_performance_vs_features.csv", index=False)

display(svm_rfe_grid_results.head())
display(rf_rfe_grid_results.head())


In [ ]:
# ============================================================
# Plot: RFE performance vs number of features
# ============================================================

def choose_best_rfe_model(grid_df):
    # Select best row by ROC-AUC first, then F1-score, then fewer features.
    return (
        grid_df.sort_values(["ROC-AUC", "F1-score", "n_features"], ascending=[False, False, True])
        .iloc[0]
        .to_dict()
    )

best_svm_rfe_row = choose_best_rfe_model(svm_rfe_grid_results)
best_rf_rfe_row = choose_best_rfe_model(rf_rfe_grid_results)


def plot_rfe_curve(grid_df, model_name_for_curve, title, out_path):
    sub = grid_df[grid_df["Algorithm"] == model_name_for_curve].sort_values("n_features")
    best = choose_best_rfe_model(sub)

    fig, ax1 = plt.subplots(figsize=(8.5, 5.2), dpi=300)
    ax2 = ax1.twinx()
    ax1.plot(sub["n_features"], sub["ROC-AUC"], marker="o", linewidth=2.5, label="ROC-AUC (test)")
    ax2.plot(sub["n_features"], sub["F1-score"], marker="s", linewidth=2.5, label="F1-score")
    ax1.axvline(best["n_features"], linestyle="--", linewidth=2)
    ax1.text(
        best["n_features"] + 1,
        best["ROC-AUC"] - 0.02,
        f"Optimum = {int(best['n_features'])}\nROC-AUC = {best['ROC-AUC']:.3f}\nF1 = {best['F1-score']:.3f}",
        fontsize=11,
        fontweight="bold",
    )
    ax1.set_xlabel("Number of features")
    ax1.set_ylabel("ROC-AUC")
    ax2.set_ylabel("F1-score")
    ax1.set_title(title, fontsize=18, fontweight="bold", color="#173f7a")
    ax1.grid(alpha=0.25)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left")
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    return best

# Requested curves. These choose the best-performing algorithm within each RFE path for clear reporting.
svm_curve_model = best_svm_rfe_row["Algorithm"]
rf_curve_model = best_rf_rfe_row["Algorithm"]

svm_curve_best = plot_rfe_curve(
    svm_rfe_grid_results,
    svm_curve_model,
    "SVM-RFE: Performance vs Number of Features",
    VALIDATION_FIGURE_DIR / "panel_05_svm_rfe_curve.png",
)
rf_curve_best = plot_rfe_curve(
    rf_rfe_grid_results,
    rf_curve_model,
    "RF-RFE: Performance vs Number of Features",
    VALIDATION_FIGURE_DIR / "panel_01_rf_rfe_curve.png",
)

print("Best SVM-RFE row:", svm_curve_best)
print("Best RF-RFE row:", rf_curve_best)


## 9.5 SVM-RFE and RF-RFE training/test performance tables

The following tables report both training and holdout-test performance, making overfitting easier to diagnose.


In [ ]:
# ============================================================
# Final RFE-path models and performance tables
# ============================================================

def get_final_rfe_models(grid_df, fitted_store, ranked_features):
    # Use the best feature count per model based on ROC-AUC/F1.
    final_models = {}
    final_features = {}
    for model_name in grid_df["Algorithm"].unique():
        sub = grid_df[grid_df["Algorithm"] == model_name]
        best = choose_best_rfe_model(sub)
        k = int(best["n_features"])
        model, features = fitted_store[(model_name, k)]
        final_models[model_name] = model
        final_features[model_name] = features
    return final_models, final_features

svm_rfe_models, svm_rfe_features_by_model = get_final_rfe_models(svm_rfe_grid_results, svm_rfe_fitted, ranked_svm_rfe_features)
rf_rfe_models, rf_rfe_features_by_model = get_final_rfe_models(rf_rfe_grid_results, rf_rfe_fitted, ranked_rf_rfe_features)

# Evaluate each model on its own selected feature subset.
def evaluate_rfe_path_models(models, feature_map, X_eval, y_eval):
    rows = []
    for name, model in models.items():
        rows.append(metric_row(name, model, X_eval[feature_map[name]], y_eval))
    return pd.DataFrame(rows)

svm_rfe_train_results = evaluate_rfe_path_models(svm_rfe_models, svm_rfe_features_by_model, X_train, y_train)
svm_rfe_test_results = evaluate_rfe_path_models(svm_rfe_models, svm_rfe_features_by_model, X_test, y_test)
rf_rfe_train_results = evaluate_rfe_path_models(rf_rfe_models, rf_rfe_features_by_model, X_train, y_train)
rf_rfe_test_results = evaluate_rfe_path_models(rf_rfe_models, rf_rfe_features_by_model, X_test, y_test)

for name, df_out in {
    "svm_rfe_train_results.csv": svm_rfe_train_results,
    "svm_rfe_test_results.csv": svm_rfe_test_results,
    "rf_rfe_train_results.csv": rf_rfe_train_results,
    "rf_rfe_test_results.csv": rf_rfe_test_results,
}.items():
    df_out.to_csv(VALIDATION_TABLE_DIR / name, index=False)

save_table_figure(
    svm_rfe_test_results,
    "SVM-RFE Test Performance",
    VALIDATION_FIGURE_DIR / "01_svm_rfe_test_results.png",
    footer="Feature-selection path: SVM-RFE; evaluation set: holdout test set.",
)
save_table_figure(
    rf_rfe_test_results,
    "RF-RFE Test Performance",
    VALIDATION_FIGURE_DIR / "02_rf_rfe_test_results.png",
    footer="Feature-selection path: RF-RFE; evaluation set: holdout test set.",
)
save_table_figure(
    svm_rfe_train_results,
    "SVM-RFE Training Performance",
    VALIDATION_FIGURE_DIR / "03_svm_rfe_train_results.png",
    footer="Feature-selection path: SVM-RFE; evaluation set: training set.",
)
save_table_figure(
    rf_rfe_train_results,
    "RF-RFE Training Performance",
    VALIDATION_FIGURE_DIR / "04_rf_rfe_train_results.png",
    footer="Feature-selection path: RF-RFE; evaluation set: training set.",
)


## 9.6 SVM-RFE and RF-RFE feature-subset tables

The following figures show the selected feature subsets for the best SVM-RFE and RF-RFE paths. These tables are intended for supplementary material or a thesis appendix.


In [ ]:
# ============================================================
# SVM-RFE and RF-RFE feature subset tables
# ============================================================

best_svm_rfe_model_name = svm_curve_best["Algorithm"]
best_rf_rfe_model_name = rf_curve_best["Algorithm"]

svm_rfe_best_features = svm_rfe_features_by_model[best_svm_rfe_model_name]
rf_rfe_best_features = rf_rfe_features_by_model[best_rf_rfe_model_name]

pd.DataFrame({"feature": svm_rfe_best_features}).to_csv(VALIDATION_TABLE_DIR / "svm_rfe_best_feature_subset.csv", index=False)
pd.DataFrame({"feature": rf_rfe_best_features}).to_csv(VALIDATION_TABLE_DIR / "rf_rfe_best_feature_subset.csv", index=False)

save_feature_subset_table(
    svm_rfe_best_features,
    "SVM-RFE Feature Subset",
    VALIDATION_FIGURE_DIR / "07_svm_rfe_feature_subset.png",
)
save_feature_subset_table(
    rf_rfe_best_features,
    "RF-RFE Feature Subset",
    VALIDATION_FIGURE_DIR / "08_rf_rfe_feature_subset.png",
)


## 9.7 Heatmap of selected features and RF-RFE feature importance

This section adds the requested **Heatmap of Selected Features** and **RF-RFE importance** visualization. The heatmap checks whether the top selected features are redundant or provide complementary information.


In [ ]:
# ============================================================
# Heatmap of Selected Features and RF-RFE importance
# ============================================================

# Use top RF-RFE features for heatmap and importance panel.
heatmap_features = rf_rfe_best_features[:10]
heatmap_data = X_model_full[heatmap_features].corr()

fig, ax = plt.subplots(figsize=(8, 7), dpi=300)
if SEABORN_AVAILABLE:
    sns.heatmap(
        heatmap_data,
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        square=True,
        cbar=True,
        ax=ax,
    )
else:
    im = ax.imshow(heatmap_data, cmap="coolwarm", vmin=-1, vmax=1)
    fig.colorbar(im, ax=ax)
    ax.set_xticks(range(len(heatmap_features)), heatmap_features, rotation=45, ha="right")
    ax.set_yticks(range(len(heatmap_features)), heatmap_features)
ax.set_title("Heatmap of Selected Features (Correlation)", fontsize=18, fontweight="bold", color="#173f7a")
fig.tight_layout()
fig.savefig(VALIDATION_FIGURE_DIR / "03_selected_features_correlation_heatmap.png", bbox_inches="tight")
plt.show()

# RF-RFE importance based on final RF model if available; otherwise use permutation importance for the best RF-RFE model.
if "RF" in rf_rfe_models:
    importance_model = rf_rfe_models["RF"]
    importance_features = rf_rfe_features_by_model["RF"]
else:
    importance_model = rf_rfe_models[best_rf_rfe_model_name]
    importance_features = rf_rfe_features_by_model[best_rf_rfe_model_name]

if hasattr(importance_model, "feature_importances_"):
    importances = importance_model.feature_importances_
else:
    perm = permutation_importance(
        importance_model,
        X_test[importance_features],
        y_test,
        scoring="roc_auc",
        n_repeats=20,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    importances = perm.importances_mean

importance_df = pd.DataFrame({
    "Feature": importance_features,
    "Importance": importances,
}).sort_values("Importance", ascending=False)
importance_df.to_csv(VALIDATION_TABLE_DIR / "rf_rfe_feature_importance.csv", index=False)

plot_imp = importance_df.head(15).sort_values("Importance")
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
ax.barh(plot_imp["Feature"], plot_imp["Importance"])
for i, v in enumerate(plot_imp["Importance"]):
    ax.text(v, i, f" {v:.3f}", va="center", fontsize=10, fontweight="bold")
ax.set_xlabel("Importance")
ax.set_title("RF-RFE: Top 15 Importance", fontsize=18, fontweight="bold", color="#173f7a")
fig.tight_layout()
fig.savefig(VALIDATION_FIGURE_DIR / "panel_02_rf_rfe_importance.png", bbox_inches="tight")
plt.show()


## 9.8 Optimized modeling with Optuna or RandomizedSearchCV fallback

Optuna is used when installed. If Optuna is unavailable, the notebook automatically falls back to `RandomizedSearchCV`. This makes the hyperparameter choices objective instead of manually fixed.


In [ ]:
# ============================================================
# Optuna / RandomizedSearchCV optimization
# ============================================================

optimized_features = rf_rfe_best_features
X_train_opt = X_train[optimized_features]
X_test_opt = X_test[optimized_features]
X_external_opt = X_external[optimized_features]

INNER_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def optimize_svm(Xtr, ytr, n_trials=30):
    if OPTUNA_AVAILABLE:
        def objective(trial):
            C = trial.suggest_float("C", 1e-3, 100, log=True)
            gamma = trial.suggest_float("gamma", 1e-4, 10, log=True)
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("model", SVC(C=C, gamma=gamma, kernel="rbf", probability=True, random_state=RANDOM_STATE)),
            ])
            return cross_val_score(model, Xtr, ytr, cv=INNER_CV, scoring="roc_auc", n_jobs=-1).mean()
        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
        params = study.best_params
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(C=params["C"], gamma=params["gamma"], kernel="rbf", probability=True, random_state=RANDOM_STATE)),
        ])
        model.fit(Xtr, ytr)
        return model, study.best_value, params

    search = RandomizedSearchCV(
        Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))]),
        param_distributions={"model__C": loguniform(1e-3, 100), "model__gamma": loguniform(1e-4, 10)},
        n_iter=n_trials,
        scoring="roc_auc",
        cv=INNER_CV,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    search.fit(Xtr, ytr)
    return search.best_estimator_, search.best_score_, search.best_params_


def optimize_rf(Xtr, ytr, n_trials=30):
    if OPTUNA_AVAILABLE:
        def objective(trial):
            model = RandomForestClassifier(
                n_estimators=trial.suggest_int("n_estimators", 200, 800),
                max_depth=trial.suggest_int("max_depth", 3, 30),
                min_samples_split=trial.suggest_int("min_samples_split", 2, 12),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 8),
                max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
            return cross_val_score(model, Xtr, ytr, cv=INNER_CV, scoring="roc_auc", n_jobs=-1).mean()
        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
        params = study.best_params
        model = RandomForestClassifier(**params, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
        model.fit(Xtr, ytr)
        return model, study.best_value, params

    search = RandomizedSearchCV(
        RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        param_distributions={
            "n_estimators": randint(200, 800),
            "max_depth": randint(3, 30),
            "min_samples_split": randint(2, 12),
            "min_samples_leaf": randint(1, 8),
            "max_features": ["sqrt", "log2", None],
        },
        n_iter=n_trials,
        scoring="roc_auc",
        cv=INNER_CV,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    search.fit(Xtr, ytr)
    return search.best_estimator_, search.best_score_, search.best_params_

svm_optimized, svm_optimized_cv, svm_optimized_params = optimize_svm(X_train_opt, y_train, n_trials=30)
rf_optimized, rf_optimized_cv, rf_optimized_params = optimize_rf(X_train_opt, y_train, n_trials=30)

optimized_models = {
    "RandomForest": rf_optimized,
    "SVM": svm_optimized,
}

optimized_test_results = evaluate_models(optimized_models, X_test_opt, y_test)
optimized_train_results = evaluate_models(optimized_models, X_train_opt, y_train)
optimized_external_results = evaluate_models(optimized_models, X_external_opt, y_external)

optimized_test_results.to_csv(VALIDATION_TABLE_DIR / "optimized_test_results.csv", index=False)
optimized_train_results.to_csv(VALIDATION_TABLE_DIR / "optimized_train_results.csv", index=False)
optimized_external_results.to_csv(VALIDATION_TABLE_DIR / "optimized_external_results.csv", index=False)

optimization_params = pd.DataFrame([
    {"model": "SVM", "cv_roc_auc": svm_optimized_cv, "params": json.dumps(svm_optimized_params)},
    {"model": "RandomForest", "cv_roc_auc": rf_optimized_cv, "params": json.dumps(rf_optimized_params)},
])
optimization_params.to_csv(VALIDATION_TABLE_DIR / "optuna_or_randomizedsearch_best_params.csv", index=False)

display(optimization_params)
display(optimized_test_results)

save_metric_barplot(
    optimized_test_results,
    "RF vs SVM Optimized Modeling",
    VALIDATION_FIGURE_DIR / "06_rf_vs_svm_optimized_modeling.png",
    metrics=("Accuracy", "F1-score", "ROC-AUC"),
    footer=f"Optimized matrix: RF-RFE best subset ({len(optimized_features)} features).",
)


## 9.9 Requested RF / SVM / RFE summary, confusion matrix, ROC curve, and probability scatter

This section selects the best final model based on holdout ROC-AUC and produces the requested summary, confusion matrix, ROC curve, and probability-vs-label visualization.


In [ ]:
# ============================================================
# Requested summary + final model diagnostic plots
# ============================================================

# Choose final model by optimized test ROC-AUC.
best_optimized_row = optimized_test_results.sort_values(["ROC-AUC", "F1-score"], ascending=False).iloc[0]
final_model_name = best_optimized_row["Algorithm"]
final_model = optimized_models[final_model_name]
final_features = optimized_features

requested_summary = pd.DataFrame([
    {
        "section": "Feature Selection Test",
        "item": "Best SVM-RFE model",
        "value": best_svm_rfe_row["Algorithm"],
        "metric": "ROC-AUC",
        "score": best_svm_rfe_row["ROC-AUC"],
    },
    {
        "section": "Feature Selection Test",
        "item": "Best RF-RFE model",
        "value": best_rf_rfe_row["Algorithm"],
        "metric": "ROC-AUC",
        "score": best_rf_rfe_row["ROC-AUC"],
    },
    {
        "section": "Baseline Modeling",
        "item": "SVM",
        "value": f"top_{baseline_k}_fscore",
        "metric": "ROC-AUC",
        "score": baseline_test_results.loc[baseline_test_results["Algorithm"] == "SVM", "ROC-AUC"].iloc[0],
    },
    {
        "section": "Baseline Modeling",
        "item": "RandomForest",
        "value": f"top_{baseline_k}_fscore",
        "metric": "ROC-AUC",
        "score": baseline_test_results.loc[baseline_test_results["Algorithm"] == "RandomForest", "ROC-AUC"].iloc[0],
    },
    {
        "section": "Optimized Modeling",
        "item": "SVM",
        "value": f"RF-RFE subset ({len(final_features)} features)",
        "metric": "ROC-AUC",
        "score": optimized_test_results.loc[optimized_test_results["Algorithm"] == "SVM", "ROC-AUC"].iloc[0],
    },
    {
        "section": "Optimized Modeling",
        "item": "RandomForest",
        "value": f"RF-RFE subset ({len(final_features)} features)",
        "metric": "ROC-AUC",
        "score": optimized_test_results.loc[optimized_test_results["Algorithm"] == "RandomForest", "ROC-AUC"].iloc[0],
    },
])
requested_summary.to_csv(VALIDATION_TABLE_DIR / "11_requested_rf_svm_rfe_summary.csv", index=False)

save_table_figure(
    requested_summary,
    "Requested RF / SVM / RFE Summary",
    VALIDATION_FIGURE_DIR / "11_rf_svm_rfe_summary_table.png",
    footer="Summary of the best model picked for each requested feature-selection path.",
    figsize=(13.5, 3.9),
)

cm_final = save_confusion_matrix_with_percent(
    final_model,
    X_test[final_features],
    y_test,
    f"Confusion Matrix for {final_model_name} with RF-RFE",
    VALIDATION_FIGURE_DIR / "09_best_rfe_confusion_matrix.png",
)
auc_final = save_roc_curve(
    final_model,
    X_test[final_features],
    y_test,
    f"ROC for {final_model_name} with RF-RFE",
    VALIDATION_FIGURE_DIR / "10_best_rfe_roc_curve.png",
)
final_probability_metrics = save_probability_scatter(
    final_model,
    X_test[final_features],
    y_test,
    "RF-RFE: Prediction Probability vs Actual Label",
    VALIDATION_FIGURE_DIR / "panel_03_rf_rfe_probability_scatter.png",
)

# Compact final summary panel.
final_summary_panel = pd.DataFrame({
    "Metric": ["Optimum features", "Final model", "ROC-AUC (test)", "F1-score", "Accuracy"],
    "Value": [len(final_features), final_model_name, best_optimized_row["ROC-AUC"], best_optimized_row["F1-score"], best_optimized_row["Accuracy"]],
})
final_summary_panel.to_csv(VALIDATION_TABLE_DIR / "panel_04_rf_rfe_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.8), dpi=300)
ax.axis("off")
ax.set_title("RF-RFE: Summary", fontsize=20, fontweight="bold", color="#1c7c3b", loc="left")
for i, row in final_summary_panel.iterrows():
    y_pos = 0.78 - i * 0.16
    ax.text(0.05, y_pos, str(row["Metric"]), fontsize=13, color="#222222")
    val = row["Value"]
    if isinstance(val, float):
        val = f"{val:.3f}"
    ax.text(0.92, y_pos, str(val), fontsize=13, fontweight="bold", color="#1c7c3b", ha="right")
    ax.hlines(y_pos - 0.06, 0.04, 0.96, color="#dce4ef", linewidth=1)
fig.savefig(VALIDATION_FIGURE_DIR / "panel_04_rf_rfe_summary.png", bbox_inches="tight")
plt.show()


## 9.10 Nested cross-validation

Nested CV estimates model generalization while accounting for hyperparameter selection inside the resampling loop. This is more rigorous than reporting a single holdout score only.


In [ ]:
# ============================================================
# Nested CV
# ============================================================

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

nested_records = []

svm_param_dist = {
    "model__C": loguniform(1e-3, 100),
    "model__gamma": loguniform(1e-4, 10),
}
rf_param_dist = {
    "n_estimators": randint(200, 700),
    "max_depth": randint(3, 25),
    "min_samples_leaf": randint(1, 8),
    "min_samples_split": randint(2, 12),
    "max_features": ["sqrt", "log2", None],
}

for model_name, base_estimator, params in [
    ("SVM", Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))]), svm_param_dist),
    ("RandomForest", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1), rf_param_dist),
]:
    outer_scores = []
    for fold_idx, (train_idx, valid_idx) in enumerate(outer_cv.split(X_model_full[final_features], y_model_full), start=1):
        Xtr, Xva = X_model_full[final_features].iloc[train_idx], X_model_full[final_features].iloc[valid_idx]
        ytr, yva = y_model_full.iloc[train_idx], y_model_full.iloc[valid_idx]
        search = RandomizedSearchCV(
            base_estimator,
            params,
            n_iter=20,
            scoring="roc_auc",
            cv=inner_cv,
            n_jobs=-1,
            random_state=RANDOM_STATE + fold_idx,
        )
        search.fit(Xtr, ytr)
        score = roc_auc_score(yva, predict_score(search.best_estimator_, Xva))
        outer_scores.append(score)
        nested_records.append({"model": model_name, "fold": fold_idx, "roc_auc": score})

nested_cv_df = pd.DataFrame(nested_records)
nested_summary = nested_cv_df.groupby("model")["roc_auc"].agg(["mean", "std", "min", "max"]).reset_index()
nested_cv_df.to_csv(VALIDATION_TABLE_DIR / "nested_cv_fold_results.csv", index=False)
nested_summary.to_csv(VALIDATION_TABLE_DIR / "nested_cv_summary.csv", index=False)
display(nested_summary)

save_table_figure(
    nested_summary.rename(columns={"model": "Model", "mean": "Mean ROC-AUC", "std": "SD", "min": "Min", "max": "Max"}),
    "Nested Cross-Validation Summary",
    VALIDATION_FIGURE_DIR / "12_nested_cv_summary.png",
    footer="Outer 5-fold CV; inner 3-fold hyperparameter search.",
)


## 9.11 Y-randomization test

Y-randomization tests whether the model performance could arise from chance correlations. The true-label model should outperform models trained on randomly permuted labels.


In [ ]:
# ============================================================
# Y-randomization test
# ============================================================

N_Y_RANDOM = 100
rng = np.random.default_rng(RANDOM_STATE)
true_auc = roc_auc_score(y_test, predict_score(final_model, X_test[final_features]))
random_aucs = []

for i in range(N_Y_RANDOM):
    y_perm = pd.Series(rng.permutation(y_train.values), index=y_train.index)
    perm_model = clone(final_model)
    perm_model.fit(X_train[final_features], y_perm)
    random_aucs.append(roc_auc_score(y_test, predict_score(perm_model, X_test[final_features])))

y_random_df = pd.DataFrame({"iteration": range(1, N_Y_RANDOM + 1), "randomized_roc_auc": random_aucs})
y_random_df.to_csv(VALIDATION_TABLE_DIR / "y_randomization_results.csv", index=False)

p_empirical = (np.sum(np.array(random_aucs) >= true_auc) + 1) / (N_Y_RANDOM + 1)

fig, ax = plt.subplots(figsize=(7, 4.5), dpi=300)
ax.hist(random_aucs, bins=20, alpha=0.75)
ax.axvline(true_auc, linewidth=2.5, linestyle="--", label=f"True ROC-AUC = {true_auc:.3f}")
ax.set_xlabel("ROC-AUC after label randomization")
ax.set_ylabel("Frequency")
ax.set_title("Y-Randomization Test", fontsize=16, fontweight="bold", color="#173f7a")
ax.legend()
fig.text(0.5, 0.02, f"Empirical p-value = {p_empirical:.4f}; iterations = {N_Y_RANDOM}", ha="center", fontsize=10)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(VALIDATION_FIGURE_DIR / "13_y_randomization_test.png", bbox_inches="tight")
plt.show()

print(f"True ROC-AUC: {true_auc:.3f}")
print(f"Randomized mean ROC-AUC: {np.mean(random_aucs):.3f} ± {np.std(random_aucs):.3f}")
print(f"Empirical p-value: {p_empirical:.4f}")


## 9.12 Applicability domain using leverage approach

The leverage-based applicability domain identifies compounds that may lie outside the chemical space represented by the training set. This is reported using the warning leverage threshold `h* = 3(p + 1) / n`, where `p` is the number of descriptors and `n` is the number of training compounds.


In [ ]:
# ============================================================
# Applicability Domain: leverage approach
# ============================================================

# To keep the leverage matrix numerically stable, use the final selected features.
Xtr_ad = X_train[final_features].copy().astype(float)
Xte_ad = X_test[final_features].copy().astype(float)

# Standardize before leverage calculation.
scaler_ad = StandardScaler()
Xtr_scaled = scaler_ad.fit_transform(Xtr_ad)
Xte_scaled = scaler_ad.transform(Xte_ad)

# Add intercept column.
Xtr_design = np.column_stack([np.ones(Xtr_scaled.shape[0]), Xtr_scaled])
Xte_design = np.column_stack([np.ones(Xte_scaled.shape[0]), Xte_scaled])

# Pseudo-inverse improves stability for correlated fingerprint bits.
H_inv = np.linalg.pinv(Xtr_design.T @ Xtr_design)
train_leverage = np.sum((Xtr_design @ H_inv) * Xtr_design, axis=1)
test_leverage = np.sum((Xte_design @ H_inv) * Xte_design, axis=1)

p = Xtr_ad.shape[1]
n = Xtr_ad.shape[0]
h_star = 3 * (p + 1) / n

# Classification residual as predicted probability minus actual label.
test_prob = predict_score(final_model, X_test[final_features])
test_residual = test_prob - y_test.values
std_resid = test_residual / (np.std(test_residual) + 1e-12)

ad_df = pd.DataFrame({
    "sample_index": X_test.index,
    "actual_label": y_test.values,
    "predicted_probability": test_prob,
    "standardized_residual": std_resid,
    "leverage": test_leverage,
    "inside_domain": test_leverage <= h_star,
})
ad_df.to_csv(VALIDATION_TABLE_DIR / "applicability_domain_leverage.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 5), dpi=300)
ax.scatter(ad_df["leverage"], ad_df["standardized_residual"], alpha=0.8)
ax.axvline(h_star, linestyle="--", linewidth=2, label=f"h* = {h_star:.3f}")
ax.axhline(3, linestyle="--", linewidth=1)
ax.axhline(-3, linestyle="--", linewidth=1)
ax.set_xlabel("Leverage")
ax.set_ylabel("Standardized residual")
ax.set_title("Applicability Domain (Williams Plot)", fontsize=16, fontweight="bold", color="#173f7a")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(VALIDATION_FIGURE_DIR / "14_applicability_domain_williams_plot.png", bbox_inches="tight")
plt.show()

print("Warning leverage h*:", h_star)
print("Compounds inside domain:", int(ad_df["inside_domain"].sum()), "/", len(ad_df))


## 9.13 External validation and bootstrap confidence intervals

The external-like validation split is used as an additional validation layer. Bootstrap confidence intervals are computed for ROC-AUC, accuracy, and F1-score to quantify uncertainty.


In [ ]:
# ============================================================
# External validation + Bootstrap CI
# ============================================================

external_metrics = pd.DataFrame([metric_row(final_model_name, final_model, X_external[final_features], y_external)])
external_metrics.to_csv(VALIDATION_TABLE_DIR / "external_validation_results.csv", index=False)
display(external_metrics)

save_table_figure(
    external_metrics,
    "External Validation Performance",
    VALIDATION_FIGURE_DIR / "15_external_validation_results.png",
    footer="External-like holdout split; replace with true external dataset when available.",
)

# Bootstrap CI on test set.
N_BOOTSTRAP = 1000
rng = np.random.default_rng(RANDOM_STATE)
y_true_arr = y_test.values
X_boot = X_test[final_features].reset_index(drop=True)
y_boot = pd.Series(y_true_arr)

boot_records = []
for b in range(N_BOOTSTRAP):
    idx = rng.integers(0, len(y_boot), size=len(y_boot))
    if len(np.unique(y_boot.iloc[idx])) < 2:
        continue
    y_b = y_boot.iloc[idx]
    X_b = X_boot.iloc[idx]
    y_pred_b = final_model.predict(X_b)
    y_score_b = predict_score(final_model, X_b)
    boot_records.append({
        "Accuracy": accuracy_score(y_b, y_pred_b),
        "F1-score": f1_score(y_b, y_pred_b, zero_division=0),
        "ROC-AUC": roc_auc_score(y_b, y_score_b),
    })

bootstrap_df = pd.DataFrame(boot_records)
bootstrap_ci = bootstrap_df.quantile([0.025, 0.5, 0.975]).T.reset_index()
bootstrap_ci.columns = ["Metric", "CI_2.5%", "Median", "CI_97.5%"]
bootstrap_df.to_csv(VALIDATION_TABLE_DIR / "bootstrap_metric_distribution.csv", index=False)
bootstrap_ci.to_csv(VALIDATION_TABLE_DIR / "bootstrap_confidence_intervals.csv", index=False)

display(bootstrap_ci)

save_table_figure(
    bootstrap_ci,
    "Bootstrap Confidence Intervals",
    VALIDATION_FIGURE_DIR / "16_bootstrap_confidence_intervals.png",
    footer=f"Bootstrap resampling on holdout test set; n = {len(bootstrap_df)} valid resamples.",
)


## 9.14 Publication note

The extended workflow now contains the requested sections and additional validation controls. For manuscript writing, the recommended result narrative is:

1. Low-variance filtering and F-score ranking reduced the descriptor space before model-dependent selection.
2. SVM-RFE and RF-RFE were compared across multiple feature-subset sizes rather than relying on a single arbitrary feature count.
3. RF and SVM were evaluated as baseline and optimized models.
4. Nested CV, external validation, y-randomization, applicability-domain analysis, and bootstrap confidence intervals were added to reduce overfitting/data-leakage concerns and strengthen model reliability.
5. The final selected model should be reported with both holdout metrics and robustness metrics rather than training performance only.
